# 01 — Punctuation Restoration Demo

This notebook demonstrates the current punctuation pipeline step-by-step using the existing code in `src/punctuation.py`.

**Model:** `Horizon-Labs/punctuation-restoration-small` (quantized ONNX, CPU inference).  
**Goal:** Add punctuation marks to Hindi text while preserving all original words and `NULL` markers.

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print('CWD:', Path.cwd())

## 1. Load the punctuation restorer

`PunctuationRestorer` loads the local tokenizer and ONNX model from `models/punctuation/` and runs inference on CPU.

In [ ]:
from src.punctuation import PunctuationRestorer, INPUT_FOLDER, OUTPUT_FOLDER

restorer = PunctuationRestorer()  # may take a moment to load
print('Input folder:', INPUT_FOLDER)
print('Output folder:', OUTPUT_FOLDER)

## 2. Inspect one input sample (before punctuation)

We use `data/model_input/sample-1.txt`. This file should be identical in content to the raw file (including `NULL` markers).

In [ ]:
sample_name = 'sample-1.txt'
text = (INPUT_FOLDER / sample_name).read_text(encoding='utf-8')
print('Characters:', len(text))
print('Literal NULL count:', text.count('NULL'))
print('\n--- Preview (first 600 chars) ---')
print(text[:600])

## 3. Restore punctuation for the sample

What happens inside `restore()` (high level):
1. Normalize whitespace
2. Remove existing punctuation characters (input is expected unpunctuated)
3. Tokenize and run the ONNX classifier to predict a label per token position
4. Insert predicted punctuation at word boundaries
5. Convert predicted `.` to Hindi danda `।`

The output is a best-effort prediction and must be reviewed manually.

In [ ]:
punctuated = restorer.restore(text)
print('Output characters:', len(punctuated))
print('Full stops (।):', punctuated.count('।'))
print('Commas (,):', punctuated.count(','))
print('Questions (?):', punctuated.count('?'))
print('Colons (:):', punctuated.count(':'))
print('Hyphens (-):', punctuated.count('-'))
print('Literal NULL count:', punctuated.count('NULL'))

In [ ]:
print('--- Punctuated preview (first 800 chars) ---')
print(punctuated[:800])

## 4. Quick integrity checks

- **NULL preserved:** literal count should match input (for this sample).
- **Words preserved:** as a rough check, compare whitespace-token counts before/after stripping punctuation characters. (Exact comparison can be tricky; always inspect manually.)

In [ ]:
import re

def rough_words(s):
    s = re.sub(r"[.,;:!?。，、；：！？؟،¿¡…।॥۔።፣፧\-]", " ", s)
    return [w for w in s.split() if w]

w_in = rough_words(text)
w_out = rough_words(punctuated)
print('Rough word/token count (input):', len(w_in))
print('Rough word/token count (output):', len(w_out))
print('Same count?', len(w_in) == len(w_out))
print('First 20 tokens in :', w_in[:20])
print('First 20 tokens out:', w_out[:20])

## 5. (Optional) Save this sample output

Uncomment to overwrite the saved file in `data/punctuated/` for this sample only.

In [ ]:
# OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
# out_path = OUTPUT_FOLDER / sample_name
# out_path.write_text(punctuated, encoding='utf-8')
# print('Saved:', out_path)

## 6. Run for all files (script)

To process every sample in `data/model_input/` and save to `data/punctuated/`, run from project root:

```powershell
python src/punctuation.py
```

After that, use `notebooks/02_null_diagnostic.ipynb` and manual inspection to evaluate quality.